In [35]:
import json
import torch
from torch.utils.data import Dataset
from collections import defaultdict
from datetime import datetime
import statistics

biomarker_keywords = {"HER2", "PD-L1", "HR", "ER", "PR"}
demographic_keywords = {"Current Age", "Sex", "Ethnicity", "Race"}

def summarize_lab_tests(lab_tests):
    cea_values = []
    ca153_values = []
    for entry in lab_tests:
        test_name = entry.get("TEST", "").strip().upper()
        try:
            value = float(entry.get("RESULT", "").strip())
        except (ValueError, TypeError):
            continue
        if test_name == "CEA":
            cea_values.append(value)
        elif test_name in {"CA_15-3", "CA15-3", "CA 15-3"}:
            ca153_values.append(value)
    lines = []
    if cea_values:
        lines.append(f"- CEA (avg): {round(statistics.mean(cea_values), 1)}")
    if ca153_values:
        lines.append(f"- CA 15-3 (avg): {round(statistics.mean(ca153_values), 1)}")
    return "Lab Tests:\n" + "\n".join(lines) if lines else ""

def extract_sample_summary(record):
    sample_data = defaultdict(dict)
    for item in record.get("CLINICAL_DATA", []):
        attr = item.get("Attribute", "").replace("(NLP)", "").strip()
        for key, value in item.items():
            if key.startswith("P-") and "-T" in key and value and value.lower() != "n/a":
                sample_data[key][attr] = value
    formatted_lines = []
    for sample_id, attributes in sample_data.items():
        parts = [f"    - {k}: {v}" for k, v in attributes.items()]
        formatted_lines.append(f"- {sample_id}:\n" + "\n".join(parts))
    return "Sample-Specific Information:\n" + "\n".join(formatted_lines) if formatted_lines else ""

def extract_treatment_summary(events, type_key, diagnosis_date=None):
    lines = []
    for t in events:
        if t.get("SUBTYPE", "").lower() == type_key:
            agent = t.get("AGENT", "").replace("(NLP)", "").strip()
            start = t.get("START_DATE")
            stop = t.get("STOP_DATE")
            if isinstance(start, int):
                start_fmt = f"{start:+d}"
            else:
                start_fmt = start
            if isinstance(stop, int):
                stop_fmt = f"{stop:+d}"
            else:
                stop_fmt = stop
            lines.append(f"{agent} ({start_fmt} to {stop_fmt})")
    return "; ".join(lines) if lines else "None"
class TreatmentOutcomeDataset(Dataset):
    def __init__(self, json_files, tokenizer=None):
        self.tokenizer = tokenizer
        self.patient_records = [(json.load(open(jf)), jf.split("/")[-1].split(".")[0]) for jf in json_files]

    def __len__(self):
        return len(self.patient_records)

    def _extract_summary(self, record):
        clinical = record.get("CLINICAL_DATA", [])
        survival_status = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival Status"), "N/A")
        survival_months = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival (Months)"), "N/A")

        demographics = []
        clinical_attrs = []
        tumor_sites = set()
        biomarkers = []

        for item in clinical:
            label = item.get("Attribute", "").replace("(NLP)", "").strip()
            value = item.get("Value", "").replace("(NLP)", "").strip()
            if not value or value.lower() == "n/a" or label in {"Overall Survival Status", "Overall Survival (Months)"}:
                continue
            if label in biomarker_keywords:
                biomarkers.append(f"- {label}: {value}")
            elif label in demographic_keywords:
                demographics.append(f"- {label}: {value}")
            elif "Tumor Site:" in label:
                if value.lower() in {"yes", "true"}:
                    site = label.split("Tumor Site:")[-1].strip()
                    tumor_sites.add(site)
            else:
                clinical_attrs.append(f"- {label}: {value}")

        blocks = []
        if demographics:
            blocks.append("Demographics:\n" + "\n".join(sorted(demographics)))
        if clinical_attrs:
            blocks.append("Clinical Attributes:\n" + "\n".join(sorted(clinical_attrs)))
        if biomarkers:
            blocks.append("Biomarkers:\n" + "\n".join(sorted(biomarkers)))
        if tumor_sites:
            blocks.append("Tumor Sites:\n" + ", ".join(sorted(tumor_sites)))

        diagnosis_date = None
        diag = record.get("Diagnosis")
        if isinstance(diag, list):
            diagnosis_date = next((i.get("DIAGNOSIS_DATE") for i in diag if i.get("DIAGNOSIS_DATE")), None)
        elif isinstance(diag, dict):
            diagnosis_date = diag.get("DIAGNOSIS_DATE")

        treatments = record.get("Treatment", [])
        blocks.append(f"Chemotherapy: {extract_treatment_summary(treatments, 'chemo', diagnosis_date)}")
        blocks.append(f"Immunotherapy: {extract_treatment_summary(treatments, 'immuno', diagnosis_date)}")
        blocks.append(f"Investigational Treatments: {extract_treatment_summary(treatments, 'investigational', diagnosis_date)}")

        therapy = record.get("therapy", [])
        radiation = "; ".join(f"{t['SUBTYPE']} starting {t['START_DATE']}" for t in therapy if "Radiation" in t.get("SUBTYPE", "")) or "None"
        blocks.append(f"Radiation Therapy: {radiation}")

        first_treat_date = next((t["START_DATE"] for t in treatments if t.get("START_DATE")), None)
        if diagnosis_date and first_treat_date:
            try:
                delta = (datetime.strptime(first_treat_date, "%Y-%m-%d") - datetime.strptime(diagnosis_date, "%Y-%m-%d")).days
                if delta >= 0:
                    blocks.append(f"Treatment Latency:\n- Time from diagnosis to first treatment: {delta} days")
            except:
                pass

        if (lab := summarize_lab_tests(record.get("LAB_TEST", []))):
            blocks.append(lab)
        if (sample := extract_sample_summary(record)):
            blocks.append(sample)

        return "\n\n".join(blocks), survival_status, survival_months

    def __getitem__(self, idx):
        record, pid = self.patient_records[idx]
        summary, status, months = self._extract_summary(record)
        prompt = (
            "You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.\n\n"
            "### TASK\n"
            "Based on the structured patient summary provided, predict the patient's survival status (living/deceased) and estimate survival duration in months.\n\n"
            "Reason clinically using disease stage, tumor sites, age, smoking history, cancer type, treatment events, lab results, and known biomarkers.\n\n"
            "### PATIENT SUMMARY\n"
            f"{summary}\n\n"
            "### OUTPUT FORMAT\n"
            "- Overall Survival Status: '0:LIVING' or '1:DECEASED'\n"
            "- Estimated Overall Survival (in months): float value\n"
        )
        if self.tokenizer:
            prompt = self.tokenizer(prompt, truncation=True, max_length=4068).input_ids
        return {"patient_data": summary, "survival_status": status, "survival_months": months, "patient_id": pid}
    
from glob import glob

patient_file = glob("patient_data_final/P-0000012.json")
dataset = TreatmentOutcomeDataset(patient_file)
sample = dataset[0]
print(sample)


with open("prompt_data_4.json", "w") as f:
    json.dump(sample, f, indent=4)

# json_file_list = glob("patient_data_final/*.json") 
# dataset = TreatmentOutcomeDataset(json_file_list)

# # Save the dataset prompts and targets as a JSON file
# output_data = []
# for i in range(len(dataset)):
#     sample = dataset[i]
#     output_data.append(sample)

# with open("prompt_data_final.json", "w") as f:
#     json.dump(output_data, f, indent=4)

# print("Dataset saved to prompt_data_final.json")


{'patient_data': 'Demographics:\n- Current Age: 68\n- Ethnicity: Non-Spanish; Non-Hispanic\n- Race: White\n- Sex: Female\n\nClinical Attributes:\n- History for Positive PD-L1: No\n- Number of Samples Per Patient: 2\n- Number of Tumor Registry Entries: 2\n- Prior Treatment to MSK: Unknown\n- Smoking History: Former/Current Smoker\n- Stage (Highest Recorded): Stage 1-3\n\nBiomarkers:\n- HER2: No\n- HR: No\n\nTumor Sites:\nIntra Abdominal, Lung, Lymph Node, Other\n\nChemotherapy: CYCLOPHOSPHAMIDE (-5437 to -5369); FLUOROURACIL (-5437 to -5326); METHOTREXATE (-5437 to -5327); CISPLATIN (33 to 40); ETOPOSIDE (33 to 65); CARBOPLATIN (61 to 68)\n\nImmunotherapy: NIVOLUMAB (1734 to 1814)\n\nInvestigational Treatments: INVESTIGATIONAL (320 to 320); INVESTIGATIONAL (320 to 320)\n\nRadiation Therapy: None\n\nLab Tests:\n- CEA (avg): 2.1\n- CA 15-3 (avg): 13.3\n\nSample-Specific Information:\n- P-0000012-T03-IM3:\n    - Cancer Type: Non-Small Cell Lung Cancer\n    - Clinical Group: 3B\n    - Clini

In [36]:
# "You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.

# ### TASK
# Given the structured patient summary, predict:
# - Overall survival status: '0:LIVING' or '1:DECEASED'
# - Estimated survival duration (in months)

# Reason clinically using patient demographics, clinical history, and all relevant sample data. If the patient has multiple cancer types or samples, prioritize more aggressive or late-stage cancers. Acknowledge uncertainty where data is missing or conflicting.

# ### PATIENT ATTRIBUTES
# Demographics:


# Clinical:


# Tumor Sites:

# ### SAMPLES OVERVIEW
# Patient has N profiled samples. Key details below:

# #### Sample ID: P-xxxx
# - Cancer Type: Lung Adenocarcinoma
# - Sample Type: Metastatic
# - TMB: 7.2
# ...

# #### Sample ID: P-yyyy
# - Cancer Type: Prostate Adenocarcinoma
# - Sample Type: Primary
# - Gleason: 9
# ...

# ### OUTPUT FORMAT
# - Overall Survival Status: ...
# - Estimated Survival Duration (months): 


In [37]:
import json
from collections import defaultdict
from datetime import datetime
import statistics

# === Load attribute metadata ===
with open("attributes_description.json") as f:
    attr_meta = json.load(f)

attr_priority = {}
attr_source = {}
for attr in attr_meta:
    label = attr["displayName"].replace("(NLP)", "").strip()
    priority = int(attr["priority"])
    attr_priority[label] = "HIGH" if priority >= 900 else "MEDIUM" if priority >= 500 else "LOW"
    attr_source[label] = "Patient" if attr["patientAttribute"] else "Sample"

# === Define cancer-specific keywords ===
cancer_specific_keywords = {
    "Breast Cancer": {"HER2", "ER", "PR", "HR", "PD-L1"},
    "Colorectal Cancer": {"MSI", "TMB", "Mutation Count"},
    "Non-Small Cell Lung Cancer": {"PD-L1", "EGFR", "ALK", "Smoking History (NLP)"},
    "Pancreatic Cancer": {"MSI", "TMB"},
    "Prostate Cancer": {"Gleason", "PSA"},
}

# === Build cancer type to relevant attributes map ===
cancer_type_attr_map = defaultdict(lambda: {"Patient": set(), "Sample": set()})
for attr, prio in attr_priority.items():
    for cancer, keywords in cancer_specific_keywords.items():
        if any(k in attr for k in keywords) or attr in {
            "Cancer Type", "Cancer Type Detailed", "Stage (Highest Recorded)",
            "Sex", "Current Age", "Overall Survival Status"
        }:
            cancer_type_attr_map[cancer][attr_source[attr]].add(attr)

# === Essential sample attributes across all cancer types ===
essential_sample_attrs = {
    "Clinical Summary", "Diagnosis Description", "ICD-O Histology Description",
    "Cancer Type Detailed", "Sample Type", "Sample Class",
    "Metastatic Site", "MSI Type", "Primary Tumor Site"
}

# === Lab and treatment helpers ===
def summarize_lab_tests(lab_tests, cancer_types):
    cea_values, ca153_values, ca19_values, psa_values = [], [], [], []
    for entry in lab_tests:
        test_name = entry.get("TEST", "").strip().upper()
        try:
            value = float(entry.get("RESULT", "").strip())
        except (ValueError, TypeError):
            continue
        if test_name == "CEA":
            cea_values.append(value)
        elif test_name in {"CA_15-3", "CA15-3", "CA 15-3"}:
            ca153_values.append(value)
        elif test_name in {"CA_19-9", "CA19-9"}:
            ca19_values.append(value)
        elif test_name == "PSA":
            psa_values.append(value)
    lines = []
    if any(ct in {"Colorectal Cancer", "Pancreatic Cancer"} for ct in cancer_types) and cea_values:
        lines.append(f"- CEA (average): {round(statistics.mean(cea_values), 1)}")
    if "Breast Cancer" in cancer_types and ca153_values:
        lines.append(f"- CA 15-3 (average): {round(statistics.mean(ca153_values), 1)}")
    if any(ct in {"Colorectal Cancer", "Pancreatic Cancer"} for ct in cancer_types) and ca19_values:
        lines.append(f"- CA 19-9 (average): {round(statistics.mean(ca19_values), 1)}")
    if "Prostate Cancer" in cancer_types and psa_values:
        lines.append(f"- PSA (average): {round(statistics.mean(psa_values), 1)}")
    return "Key Tumor Markers:\n" + "\n".join(lines) if lines else ""

def extract_treatment_summary(events, type_key, label):
    lines = []
    dates = []
    for t in events:
        if t.get("SUBTYPE", "").lower() == type_key:
            agent = t.get("AGENT", "").replace("(NLP)", "").strip()
            start = t.get("START_DATE")
            stop = t.get("STOP_DATE")
            dates.append((start, stop))
            lines.append(agent.upper() if agent else "[Unknown Agent]")
    if not lines:
        return f"{label}: None"
    agents = ", ".join(sorted(set(lines)))
    all_dates = [int(d) for pair in dates for d in pair if d and isinstance(d, (int, str)) and str(d).isdigit()]
    date_range = f"from {min(all_dates)} to {max(all_dates)}" if all_dates else "(date unknown)"
    return f"{label}: {agents}\n{label} Timeline (days relative to diagnosis): {date_range}"

# === Prompt templates ===
cancer_prompt_templates = {
    "Breast Cancer": "The patient has been diagnosed with Breast Cancer. Use clinical, treatment, and genomic information to predict their survival outcome.",
    "Non-Small Cell Lung Cancer": "The patient has Non-Small Cell Lung Cancer. Carefully consider PD-L1 status and genomic mutations in your prediction.",
    "Colorectal Cancer": "The patient has Colorectal Cancer. Incorporate MSI, TMB, and biomarker data to estimate survival duration.",
    "Pancreatic Cancer": "The patient has Pancreatic Cancer. Consider stage and relevant mutations for prognosis.",
    "Prostate Cancer": "This patient has Prostate Cancer. Evaluate based on PSA levels, Gleason score, and treatment data."
}

def generate_patient_prompt(record):
    clinical = record.get("CLINICAL_DATA", [])
    survival_status = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival Status"), "N/A")
    survival_months = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival (Months)"), "N/A")

    cancer_type_flags = set()
    sample_data = defaultdict(dict)
    for item in clinical:
        attr = item.get("Attribute", "").replace("(NLP)", "").strip()
        for k, v in item.items():
            if k.startswith("P-") and "-T" in k and v and v.lower() != "n/a":
                sample_data[k][attr] = v
                if attr == "Cancer Type":
                    cancer_type_flags.add(v)

    attr_whitelist = {"Patient": set(), "Sample": set()}
    for ct in cancer_type_flags:
        if ct in cancer_type_attr_map:
            for scope in ["Patient", "Sample"]:
                attr_whitelist[scope].update(cancer_type_attr_map[ct][scope])

    demographics, clinical_attrs, tumor_sites, biomarkers = [], [], set(), []
    for item in clinical:
        label = item.get("Attribute", "").replace("(NLP)", "").strip()
        value = item.get("Value", "").replace("(NLP)", "").strip()
        if not value or value.lower() == "n/a" or label in {"Overall Survival Status", "Overall Survival (Months)", "Number of Tumor Registry Entries", "Number of Samples Per Patient", "Race", "Ethnicity"}:
            continue
        if label in {"HER2", "PD-L1", "HR", "ER", "PR"}:
            if label == "PD-L1" or ("Breast Cancer" in cancer_type_flags and label != "PD-L1"):
                biomarkers.append(f"- {label}: {value}")
        elif "Tumor Site:" in label and value.lower() in {"yes", "true"}:
            tumor_sites.add(label.split("Tumor Site:")[-1].strip())
        elif label == "Smoking History (NLP)" and "Non-Small Cell Lung Cancer" in cancer_type_flags:
            clinical_attrs.append(f"- {label}: {value}")
        else:
            src = attr_source.get(label, "Patient")
            if label in {"Current Age", "Sex"}:
                demographics.append(f"- {label}: {value}")
            elif label in attr_whitelist[src] or label in essential_sample_attrs:
                clinical_attrs.append(f"- {label}: {value}")

    blocks = []
    if demographics:
        blocks.append("Demographics:\n" + "\n".join(sorted(demographics)))
    if clinical_attrs:
        blocks.append("Clinical Attributes:\n" + "\n".join(sorted(clinical_attrs)))
    if biomarkers:
        blocks.append("Biomarkers:\n" + "\n".join(sorted(biomarkers)))
    if tumor_sites:
        blocks.append("Tumor Sites:\n" + ", ".join(sorted(tumor_sites)))

    treatments = record.get("Treatment", [])
    extra_therapy = record.get("TREATMENT", [])
    all_radiation = [t for t in (treatments + extra_therapy) if "Radiation" in t.get("SUBTYPE", "")]
    radiation = "; ".join(f"{t.get('SUBTYPE')} starting {t.get('START_DATE')}" for t in all_radiation) or "None"

    blocks.append(extract_treatment_summary(treatments, "chemo", "Chemotherapy"))
    blocks.append(extract_treatment_summary(treatments, "immuno", "Immunotherapy"))
    blocks.append(extract_treatment_summary(treatments, "investigational", "Investigational Treatments"))
    blocks.append(f"Radiation Therapy: {radiation}")

    if (lab := summarize_lab_tests(record.get("LAB_TEST", []), cancer_type_flags)):
        blocks.append(lab)

    if len(cancer_type_flags) > 1:
        blocks.append("\n### MULTIPLE CANCER TYPES DETECTED")
        blocks.append("The patient has confirmed diagnoses of:")
        for ct in cancer_type_flags:
            blocks.append(f"- {ct}")

    if sample_data:
        blocks.append("\nSample-Specific Information:")
        cancer_type_to_samples = defaultdict(list)
        for sid, attrs in sample_data.items():
            ct = attrs.get("Cancer Type", "Unknown Cancer")
            cancer_type_to_samples[ct].append((sid, attrs))

        for ct, samples in cancer_type_to_samples.items():
            for sid, attrs in samples:
                blocks.append(f"\n{ct} Sample Summary:")
                selected = {k: v for k, v in attrs.items() if k in essential_sample_attrs}
                for k, v in selected.items():
                    blocks.append(f"    - {k}: {v}")

    patient_summary = "\n\n".join(blocks)

    if len(cancer_type_flags) == 1:
        (single_cancer,) = cancer_type_flags
        task_note = cancer_prompt_templates.get(single_cancer, f"The patient has been diagnosed with {single_cancer}. Predict survival using all relevant attributes.")
    else:
        task_note = (
            "If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer "
            "based on available stage, treatment history, or mutation burden."
        )

    prompt = (
        "You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.\n\n"
        "### TASK\n"
        "Based on the structured patient summary provided, predict the patient's survival status (living/deceased) and estimate survival duration in months.\n\n"
        f"{task_note}\n\n"
        "### PATIENT SUMMARY\n"
        f"{patient_summary}\n\n"
        "### OUTPUT FORMAT\n"
        "- Overall Survival Status: '0:LIVING' or '1:DECEASED'\n"
        "- Estimated Overall Survival (in months): float value\n"
    )
    return prompt, survival_status, survival_months







with open("patient_data_final/P-0000012.json") as f:
    record = json.load(f)

prompt, status, months = generate_patient_prompt(record)
print(prompt)



You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.

### TASK
Based on the structured patient summary provided, predict the patient's survival status (living/deceased) and estimate survival duration in months.

If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer based on available stage, treatment history, or mutation burden.

### PATIENT SUMMARY
Demographics:
- Current Age: 68
- Sex: Female

Clinical Attributes:
- History for Positive PD-L1: No
- Stage (Highest Recorded): Stage 1-3

Biomarkers:
- HER2: No
- HR: No

Tumor Sites:
Intra Abdominal, Lung, Lymph Node, Other

Chemotherapy: CARBOPLATIN, CISPLATIN, CYCLOPHOSPHAMIDE, ETOPOSIDE, FLUOROURACIL, METHOTREXATE
Chemotherapy Timeline (days relative to diagnosis): from 33 to 68

Immunotherapy: NIVOLUMAB
Immunotherapy Timeline (days relative to diagnosis): from 1734 to 1814

Investigational Treatments: INVESTIGATIONAL
Investigational Tr

In [15]:
import json
from collections import defaultdict
from datetime import datetime
import statistics

# === Load attribute metadata ===
with open("attributes_description.json") as f:
    attr_meta = json.load(f)

attr_priority = {}
attr_source = {}
for attr in attr_meta:
    label = attr["displayName"].replace("(NLP)", "").strip()
    priority = int(attr["priority"])
    attr_priority[label] = "HIGH" if priority >= 900 else "MEDIUM" if priority >= 500 else "LOW"
    attr_source[label] = "Patient" if attr["patientAttribute"] else "Sample"

# === Define cancer-specific keywords ===
cancer_specific_keywords = {
    "Breast Cancer": {"HER2", "ER", "PR", "HR"},
    "Colorectal Cancer": {"MSI", "TMB", "Mutation Count"},
    "Non-Small Cell Lung Cancer": {"Smoking", "PD-L1", "EGFR", "ALK"},
    "Pancreatic Cancer": {"MSI", "TMB"},
    "Prostate Cancer": {"Gleason", "PSA"},
}

# === Build cancer type to relevant attributes map ===
cancer_type_attr_map = defaultdict(lambda: {"Patient": set(), "Sample": set()})
for attr, prio in attr_priority.items():
    if prio in {"HIGH", "MEDIUM"}:
        for cancer, keywords in cancer_specific_keywords.items():
            if any(k in attr for k in keywords) or attr in {
                "Cancer Type", "Cancer Type Detailed", "Stage (Highest Recorded)",
                "Sex", "Race", "Ethnicity", "Current Age", "Smoking History (NLP)", "Overall Survival Status"
            }:
                cancer_type_attr_map[cancer][attr_source[attr]].add(attr)

# === Essential sample attributes across all cancer types ===
essential_sample_attrs = {
    "Cancer Type", "Cancer Type Detailed", "Sample Type",
    "TMB (nonsynonymous)", "MSI Score", "Mutation Count",
    "Fraction Genome Altered", "Primary Tumor Site"
}

# === Lab and treatment helpers ===
def summarize_lab_tests(lab_tests):
    cea_values, ca153_values = [], []
    for entry in lab_tests:
        test_name = entry.get("TEST", "").strip().upper()
        try:
            value = float(entry.get("RESULT", "").strip())
        except (ValueError, TypeError):
            continue
        if test_name == "CEA":
            cea_values.append(value)
        elif test_name in {"CA_15-3", "CA15-3", "CA 15-3"}:
            ca153_values.append(value)
    lines = []
    if cea_values:
        lines.append(f"- CEA (average): {round(statistics.mean(cea_values), 1)}")
    if ca153_values:
        lines.append(f"- CA 15-3 (average): {round(statistics.mean(ca153_values), 1)}")
    return "Key Tumor Markers:\n" + "\n".join(lines) if lines else ""

def extract_treatment_summary(events, type_key, label):
    lines = []
    dates = []
    for t in events:
        if t.get("SUBTYPE", "").lower() == type_key:
            agent = t.get("AGENT", "").replace("(NLP)", "").strip()
            start = t.get("START_DATE")
            stop = t.get("STOP_DATE")
            dates.append((start, stop))
            lines.append(agent.upper() if agent else "[Unknown Agent]")
    if not lines:
        return f"{label}: None"
    agents = ", ".join(sorted(set(lines)))
    all_dates = [int(d) for pair in dates for d in pair if d and isinstance(d, (int, str)) and str(d).isdigit()]
    date_range = f"from {min(all_dates)} to {max(all_dates)}" if all_dates else "(date unknown)"
    return f"{label}: {agents}\n{label} Timeline (days relative to diagnosis): {date_range}"

# === Prompt templates ===
cancer_prompt_templates = {
    "Breast Cancer": "The patient has been diagnosed with Breast Cancer. Use clinical, treatment, and genomic information to predict their survival outcome.",
    "Non-Small Cell Lung Cancer": "The patient has Non-Small Cell Lung Cancer. Carefully consider smoking history, PD-L1 status, and genomic mutations in your prediction.",
    "Colorectal Cancer": "The patient has Colorectal Cancer. Incorporate MSI, TMB, and biomarker data to estimate survival duration.",
    "Pancreatic Cancer": "The patient has Pancreatic Cancer. Consider stage and relevant mutations for prognosis.",
    "Prostate Cancer": "This patient has Prostate Cancer. Evaluate based on PSA levels, Gleason score, and treatment data."
}

def generate_patient_prompt(record):
    clinical = record.get("CLINICAL_DATA", [])
    survival_status = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival Status"), "N/A")
    survival_months = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival (Months)"), "N/A")

    cancer_types = defaultdict(lambda: {"Sample": set()})
    sample_data = defaultdict(dict)
    for item in clinical:
        attr = item.get("Attribute", "").replace("(NLP)", "").strip()
        for k, v in item.items():
            if k.startswith("P-") and "-T" in k and v and v.lower() != "n/a":
                sample_data[k][attr] = v
                if attr == "Cancer Type":
                    cancer_types[v]["Sample"].add(k)

    attr_whitelist = {"Patient": set(), "Sample": set()}
    for ct in cancer_types:
        if ct in cancer_type_attr_map:
            for scope in ["Patient", "Sample"]:
                attr_whitelist[scope].update(cancer_type_attr_map[ct][scope])

    demographics, clinical_attrs, tumor_sites, biomarkers = [], [], set(), []
    for item in clinical:
        label = item.get("Attribute", "").replace("(NLP)", "").strip()
        value = item.get("Value", "").replace("(NLP)", "").strip()
        if not value or value.lower() == "n/a" or label in {"Overall Survival Status", "Overall Survival (Months)"}:
            continue
        src = attr_source.get(label, "Patient")
        if label not in attr_whitelist[src]:
            continue
        if any(label in cancer_type_attr_map[ct]["Patient"] for ct in cancer_types):
            if label in {"HER2", "PD-L1", "HR", "ER", "PR"}:
                biomarkers.append(f"- {label}: {value}")
        if label in {"Current Age", "Sex", "Ethnicity", "Race"}:
            demographics.append(f"- {label}: {value}")
        elif "Tumor Site:" in label and value.lower() in {"yes", "true"}:
            tumor_sites.add(label.split("Tumor Site:")[-1].strip())
        else:
            clinical_attrs.append(f"- {label}: {value}")

    blocks = []
    if demographics:
        blocks.append("Demographics:\n" + "\n".join(sorted(demographics)))
    if clinical_attrs:
        blocks.append("Clinical Attributes:\n" + "\n".join(sorted(clinical_attrs)))
    if biomarkers:
        blocks.append("Biomarkers:\n" + "\n".join(sorted(biomarkers)))
    if tumor_sites:
        blocks.append("Tumor Sites:\n" + ", ".join(sorted(tumor_sites)))

    treatments = record.get("Treatment", [])
    extra_therapy = record.get("TREATMENT", [])
    all_radiation = [t for t in (treatments + extra_therapy) if "Radiation" in t.get("SUBTYPE", "")]
    radiation = "; ".join(f"{t.get('SUBTYPE')} starting {t.get('START_DATE')}" for t in all_radiation) or "None"

    blocks.append(extract_treatment_summary(treatments, "chemo", "Chemotherapy"))
    blocks.append(extract_treatment_summary(treatments, "immuno", "Immunotherapy"))
    blocks.append(extract_treatment_summary(treatments, "investigational", "Investigational Treatments"))
    blocks.append(f"Radiation Therapy: {radiation}")

    if (lab := summarize_lab_tests(record.get("LAB_TEST", []))):
        blocks.append(lab)

    if len(cancer_types) > 1:
        blocks.append("\n### MULTIPLE CANCER TYPES DETECTED")
        blocks.append("The patient has confirmed diagnoses of:")
        for ct in cancer_types:
            blocks.append(f"- {ct}")

    if sample_data:
        blocks.append("\nSample-Specific Information:")
        for ct, samples in cancer_types.items():
            for sid in sorted(samples["Sample"]):
                if sid in sample_data:
                    attrs = sample_data[sid]
                    selected = {k: v for k, v in attrs.items() if k in attr_whitelist["Sample"] or k in essential_sample_attrs}
                    blocks.append(f"\nSample Summary ({ct}):")
                    for k, v in selected.items():
                        if k not in {"Cancer Type"}:
                            blocks.append(f"    - {k}: {v}")

    patient_summary = "\n\n".join(blocks)

    if len(cancer_types) == 1:
        (single_cancer,) = cancer_types.keys()
        task_note = cancer_prompt_templates.get(single_cancer, f"The patient has been diagnosed with {single_cancer}. Predict survival using all relevant attributes.")
    else:
        task_note = (
            "If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer "
            "based on available stage, treatment history, or mutation burden."
        )

    prompt = (
        "You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.\n\n"
        "### TASK\n"
        "Based on the structured patient summary provided, predict the patient's survival status (living/deceased) and estimate survival duration in months.\n\n"
        f"{task_note}\n\n"
        "### PATIENT SUMMARY\n"
        f"{patient_summary}\n\n"
        "### OUTPUT FORMAT\n"
        "- Overall Survival Status: '0:LIVING' or '1:DECEASED'\n"
        "- Estimated Overall Survival (in months): float value\n"
    )
    return prompt, survival_status, survival_months



with open("patient_data_final/P-0000012.json") as f:
    record = json.load(f)

prompt, status, months = generate_patient_prompt(record)
print(prompt)


You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.

### TASK
Based on the structured patient summary provided, predict the patient's survival status (living/deceased) and estimate survival duration in months.

If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer based on available stage, treatment history, or mutation burden.

### PATIENT SUMMARY
Demographics:
- Current Age: 68
- Ethnicity: Non-Spanish; Non-Hispanic
- Race: White
- Sex: Female

Clinical Attributes:
- History for Positive PD-L1: No
- Smoking History: Former/Current Smoker
- Stage (Highest Recorded): Stage 1-3

Chemotherapy: CARBOPLATIN, CISPLATIN, CYCLOPHOSPHAMIDE, ETOPOSIDE, FLUOROURACIL, METHOTREXATE
Chemotherapy Timeline (days relative to diagnosis): from 33 to 68

Immunotherapy: NIVOLUMAB
Immunotherapy Timeline (days relative to diagnosis): from 1734 to 1814

Investigational Treatments: INVESTIGATIONAL
Investigatio

In [2]:
print("Demographics:\n- Current Age: 68\n- Ethnicity: Non-Spanish; Non-Hispanic\n- Race: White\n- Sex: Female\n\nClinical Attributes:\n- History for Positive PD-L1: No\n- Number of Samples Per Patient: 2\n- Number of Tumor Registry Entries: 2\n- Prior Treatment to MSK: Unknown\n- Smoking History: Former/Current Smoker\n- Stage (Highest Recorded): Stage 1-3\n\nBiomarkers:\n- HER2: No\n- HR: No\n\nTumor Sites:\nIntra Abdominal, Lung, Lymph Node, Other\n\nChemotherapy: CYCLOPHOSPHAMIDE (-5437 to -5369); FLUOROURACIL (-5437 to -5326); METHOTREXATE (-5437 to -5327); CISPLATIN (33 to 40); ETOPOSIDE (33 to 65); CARBOPLATIN (61 to 68)\n\nImmunotherapy: NIVOLUMAB (1734 to 1814)\n\nInvestigational Treatments: INVESTIGATIONAL (320 to 320); INVESTIGATIONAL (320 to 320)\n\nRadiation Therapy: None\n\nLab Tests:\n- CEA (avg): 2.1\n- CA 15-3 (avg): 13.3\n\nSample-Specific Information:\n- P-0000012-T03-IM3:\n    - Cancer Type: Non-Small Cell Lung Cancer\n    - Clinical Group: 3B\n    - Clinical Summary: Distant\n    - Diagnosis Description: Lung and Bronchus\n    - ICD-O Histology Description: Adenocarcinoma, Nos\n    - Cancer Type Detailed: Lung Adenocarcinoma\n    - Sample Type: Metastasis\n    - MSI Type: Stable\n    - Gene Panel: IMPACT341\n    - Somatic Status: Matched\n    - Sample Class: Tumor\n    - Mutation Count: 30\n    - Fraction Genome Altered: 0.1844\n    - TMB (nonsynonymous): 32.16550372\n    - Metastatic Site: Neck\n    - MSI Comment: MICROSATELLITE STABLE (MSS). See MSI note below.\n    - MSI Score: 0.47\n    - Oncotree Code: LUAD\n    - Primary Tumor Site: Lung\n    - Sample coverage: 428\n- P-0000012-T02-IM3:\n    - Cancer Type: Breast Cancer\n    - Diagnosis Description: Breast\n    - ICD-O Histology Description: Infiltrating Duct Carcinoma\n    - Cancer Type Detailed: Breast Invasive Ductal Carcinoma\n    - Sample Type: Primary\n    - MSI Type: Indeterminate\n    - Gene Panel: IMPACT341\n    - Somatic Status: Matched\n    - Sample Class: Tumor\n    - Mutation Count: 1\n    - Fraction Genome Altered: 0.3146\n    - TMB (nonsynonymous): 1.109155301\n    - MSI Comment: MICROSATELLITE INSTABILITY-INDETERMINATE. See MSI note below.\n    - MSI Score: 4.1\n    - Oncotree Code: IDC\n    - Primary Tumor Site: Breast\n    - Sample coverage: 344")

Demographics:
- Current Age: 68
- Ethnicity: Non-Spanish; Non-Hispanic
- Race: White
- Sex: Female

Clinical Attributes:
- History for Positive PD-L1: No
- Number of Samples Per Patient: 2
- Number of Tumor Registry Entries: 2
- Prior Treatment to MSK: Unknown
- Smoking History: Former/Current Smoker
- Stage (Highest Recorded): Stage 1-3

Biomarkers:
- HER2: No
- HR: No

Tumor Sites:
Intra Abdominal, Lung, Lymph Node, Other

Chemotherapy: CYCLOPHOSPHAMIDE (-5437 to -5369); FLUOROURACIL (-5437 to -5326); METHOTREXATE (-5437 to -5327); CISPLATIN (33 to 40); ETOPOSIDE (33 to 65); CARBOPLATIN (61 to 68)

Immunotherapy: NIVOLUMAB (1734 to 1814)

Investigational Treatments: INVESTIGATIONAL (320 to 320); INVESTIGATIONAL (320 to 320)

Radiation Therapy: None

Lab Tests:
- CEA (avg): 2.1
- CA 15-3 (avg): 13.3

Sample-Specific Information:
- P-0000012-T03-IM3:
    - Cancer Type: Non-Small Cell Lung Cancer
    - Clinical Group: 3B
    - Clinical Summary: Distant
    - Diagnosis Description: Lung 

In [4]:
import json
from collections import defaultdict
from datetime import datetime
import statistics

# === Load attribute metadata ===
with open("attributes_description.json") as f:
    attr_meta = json.load(f)

attr_priority = {}
attr_source = {}
for attr in attr_meta:
    label = attr["displayName"].replace("(NLP)", "").strip()
    priority = int(attr["priority"])
    attr_priority[label] = "HIGH" if priority >= 900 else "MEDIUM" if priority >= 500 else "LOW"
    attr_source[label] = "Patient" if attr["patientAttribute"] else "Sample"

# === Define cancer-specific keywords ===
cancer_specific_keywords = {
    "Breast Cancer": {"HER2", "ER", "PR", "HR", "PD-L1"},
    "Colorectal Cancer": {"MSI", "TMB", "Mutation Count"},
    "Non-Small Cell Lung Cancer": {"PD-L1", "EGFR", "ALK", "Smoking History (NLP)", "Smoking History"},
    "Pancreatic Cancer": {"MSI", "TMB"},
    "Prostate Cancer": {"Gleason", "PSA"},
}

# === Build cancer type to relevant attributes map ===
cancer_type_attr_map = defaultdict(lambda: {"Patient": set(), "Sample": set()})
for attr, prio in attr_priority.items():
    for cancer, keywords in cancer_specific_keywords.items():
        if any(k in attr for k in keywords) or attr in {
            "Cancer Type", "Cancer Type Detailed", "Stage (Highest Recorded)",
            "Sex", "Current Age", "Overall Survival Status", "Smoking History","Smoking History (NLP)"
        }:
            cancer_type_attr_map[cancer][attr_source[attr]].add(attr)

# === Sample attributes for each cancer ===
sample_attrs_by_cancer = defaultdict(lambda: {
    "Clinical Summary", "Diagnosis Description", "ICD-O Histology Description",
    "Cancer Type Detailed", "Sample Type",
    "Metastatic Site", "MSI Type", "Primary Tumor Site", "Clinical Group"
})
sample_attrs_by_cancer["Breast Cancer"] = {
    "Cancer Type Detailed", "Sample Type", "Metastatic Site"
}

sample_attrs_by_cancer["Non-Small Cell Lung Cancer"] = {
    "Cancer Type Detailed", "Sample Type", 
    "Metastatic Site", "MSI Type", "MSI Score", "Clinical Group"
}

sample_attrs_by_cancer["Colorectal Cancer"] = {
    "Clinical Summary", "Diagnosis Description","Cancer Type Detailed", "Sample Type", "Clinical Group"
    "Metastatic Site", "MSI Type", "Primary Tumor Site"
}

sample_attrs_by_cancer["Prostate Cancer"] = {
    "Clinical Summary", "Cancer Type Detailed", "Sample Type", "Clinical Group"
    "Metastatic Site"
}

sample_attrs_by_cancer["Pancreatic Cancer"] = {
    "Clinical Summary", "ICD-O Histology Description",
    "Cancer Type Detailed", "Sample Type",
    "Metastatic Site", "MSI Type",  "Clinical Group"
}

# def summarize_lab_tests(lab_tests, cancer_types):
#     cea_values, ca153_values, ca19_values, psa_values = [], [], [], []
#     for entry in lab_tests:
#         test_name = entry.get("TEST", "").strip().upper()
#         try:
#             value = float(entry.get("RESULT", "").strip())
#         except (ValueError, TypeError):
#             continue
#         if test_name == "CEA":
#             cea_values.append(value)
#         elif test_name in {"CA_15-3", "CA15-3", "CA 15-3"}:
#             ca153_values.append(value)
#         elif test_name in {"CA_19-9", "CA19-9"}:
#             ca19_values.append(value)
#         elif test_name == "PSA":
#             psa_values.append(value)
#     markers = []
#     if any(ct in {"Colorectal Cancer", "Pancreatic Cancer"} for ct in cancer_types) and cea_values:
#         markers.append(f"CEA={round(statistics.mean(cea_values), 1)}")
#     if "Breast Cancer" in cancer_types and ca153_values:
#         markers.append(f"CA15-3={round(statistics.mean(ca153_values), 1)}")
#     if any(ct in {"Colorectal Cancer", "Pancreatic Cancer"} for ct in cancer_types) and ca19_values:
#         markers.append(f"CA19-9={round(statistics.mean(ca19_values), 1)}")
#     if "Prostate Cancer" in cancer_types and psa_values:
#         markers.append(f"PSA={round(statistics.mean(psa_values), 1)}")
#     return "Key Tumor Markers: " + "; ".join(markers) if markers else ""

from scipy.stats import linregress

def summarize_lab_tests(lab_tests, cancer_types):
    import statistics

    def compute_trend(values):
        if len(values) < 2:
            return None  # Not enough data
        values.sort(key=lambda x: x[0])  # Sort by date
        days = [v[0] for v in values]
        results = [v[1] for v in values]
        if len(set(days)) == 1:
            return "constant"
        slope, *_ = linregress(days, results)
        direction = "rising" if slope > 0 else "falling" if slope < 0 else "stable"
        return f"{direction}, slope={round(slope, 3)}"


    def extract_values(test_name_aliases):
        return [(int(entry.get("DAYS_FROM_DIAGNOSIS", 0)), float(entry["RESULT"]))
                for entry in lab_tests
                if entry.get("TEST", "").strip().upper() in test_name_aliases
                and entry.get("RESULT", "").replace('.', '', 1).isdigit()]

    cea_values = extract_values({"CEA"})
    ca153_values = extract_values({"CA_15-3", "CA15-3", "CA 15-3"})
    ca19_values = extract_values({"CA_19-9", "CA19-9"})
    psa_values = extract_values({"PSA"})

    markers = []

    if any(ct in {"Colorectal Cancer", "Pancreatic Cancer", "Non-Small Cell Lung Cancer"} for ct in cancer_types) and cea_values:
        trend = compute_trend(cea_values)
        mean_val = round(statistics.mean(v for _, v in cea_values), 1)
        markers.append(f"CEA={mean_val} ({trend})" if trend else f"CEA={mean_val}")

    if "Breast Cancer" in cancer_types and ca153_values:
        trend = compute_trend(ca153_values)
        mean_val = round(statistics.mean(v for _, v in ca153_values), 1)
        markers.append(f"CA15-3={mean_val} ({trend})" if trend else f"CA15-3={mean_val}")

    if any(ct in {"Colorectal Cancer", "Pancreatic Cancer"} for ct in cancer_types) and ca19_values:
        trend = compute_trend(ca19_values)
        mean_val = round(statistics.mean(v for _, v in ca19_values), 1)
        markers.append(f"CA19-9={mean_val} ({trend})" if trend else f"CA19-9={mean_val}")

    if "Prostate Cancer" in cancer_types and psa_values:
        trend = compute_trend(psa_values)
        mean_val = round(statistics.mean(v for _, v in psa_values), 1)
        markers.append(f"PSA={mean_val} ({trend})" if trend else f"PSA={mean_val}")

    return "Key Tumor Markers:\n" + "; ".join(markers) if markers else ""


def extract_treatment_summary(events, type_key, label):
    lines = []
    dates = []
    for t in events:
        if t.get("SUBTYPE", "").lower() == type_key:
            agent = t.get("AGENT", "").replace("(NLP)", "").strip()
            start = t.get("START_DATE")
            stop = t.get("STOP_DATE")
            dates.append((start, stop))
            lines.append(agent.upper() if agent else "[Unknown Agent]")
    if not lines:
        return ""
    agents = ", ".join(sorted(set(lines)))
    all_dates = [int(d) for pair in dates for d in pair if d and isinstance(d, (int, str)) and str(d).isdigit()]
    date_range = f"Days: {min(all_dates)}–{max(all_dates)}" if all_dates else "Days: Unknown"
    return f"- {label}: {agents}, {date_range}"


# def extract_surgery_summary(events):
#     lines = []
#     dates = []
#     for t in events:
#         if t.get("SUBTYPE", "").lower() == "procedure":
#             start = t.get("START_DATE")
#             dates.append(start)
#             lines.append("Procedure")
#     if not lines:
#         return "- Surgery: None"
#     all_dates = [int(d) for d in dates if d and isinstance(d, (int, str)) and str(d).isdigit()]
#     date_range = f"Days: {min(all_dates)}–{max(all_dates)}" if all_dates else "Days: Unknown"
#     return f"- Surgery: Procedure, {date_range}"

# === Prompt templates ===
cancer_prompt_templates = {
    "Breast Cancer": "The patient has been diagnosed with Breast Cancer. Use your clinical reasoning to interpret the data attributes related to Cancer Stage, Age, tumor biology, treatments, biomarkers, and metastasis patterns.",
    "Non-Small Cell Lung Cancer": "The patient has Non-Small Cell Lung Cancer. Focus on relevant clinical factors such as Cancer Stage, Age, smoking history, and metastasis patterns.",
    "Colorectal Cancer": "The patient has Colorectal Cancer. Focus on relevant clinical factors such as Cancer Stage, Age, Tumor markers, Treatment data and metastasis patterns.",
    "Pancreatic Cancer": "The patient has Pancreatic Cancer. Consider stage and relevant mutations for prognosis.",
    "Prostate Cancer": "The patient has Prostate Cancer. Focus on relevant key factors such as Cancer Stage, Age, Gleason score, PSA levels, and treatment data."
}

def generate_patient_prompt(record):
    clinical = record.get("CLINICAL_DATA", [])
    survival_status = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival Status"), "N/A")
    survival_months = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival (Months)"), "N/A")

    cancer_type_flags = set()
    sample_data = defaultdict(dict)
    for item in clinical:
        attr = item.get("Attribute", "").replace("(NLP)", "").strip()
        for k, v in item.items():
            if k.startswith("P-") and "-T" in k and v and v.lower() != "n/a":
                sample_data[k][attr] = v
                if attr == "Cancer Type":
                    cancer_type_flags.add(v)

    attr_whitelist = {"Patient": set(), "Sample": set()}
    for ct in cancer_type_flags:
        if ct in cancer_type_attr_map:
            for scope in ["Patient", "Sample"]:
                attr_whitelist[scope].update(cancer_type_attr_map[ct][scope])

    demographics, clinical_attrs, tumor_sites, biomarkers = [], [], set(), []
    for item in clinical:
        label = item.get("Attribute", "").replace("(NLP)", "").strip()
        value = item.get("Value", "").replace("(NLP)", "").strip()
        if not value or value.lower() == "n/a" or label in {"Overall Survival Status", "Overall Survival (Months)", "Number of Tumor Registry Entries", "Number of Samples Per Patient", "Race", "Ethnicity"}:
            continue
        if label in {"HER2", "PD-L1", "HR", "ER", "PR"}:
            if label == "PD-L1" or ("Breast Cancer" in cancer_type_flags and label != "PD-L1"):
                biomarkers.append(f"{label}={value}")
        elif "Tumor Site:" in label and value.lower() in {"yes", "true"}:
            tumor_sites.add(label.split("Tumor Site:")[-1].strip())
        elif label == "Smoking History" and "Non-Small Cell Lung Cancer" in cancer_type_flags:
            clinical_attrs.append(f"Smoking={value}")
        elif label == "Smoking History" and "Non-Small Cell Lung Cancer" not in cancer_type_flags:
            continue
        elif label == "Stage (Highest Recorded)":
            clinical_attrs.append(f"Cancer {value}")
        else:
            src = attr_source.get(label, "Patient")
            if label in {"Current Age"}:
                demographics.append(f"{label}={value}")
            elif label in {"Sex"}:
                if ("Non-Small Cell Lung Cancer" in cancer_type_flags or "Colorectal Cancer" in cancer_type_flags or "Pancreatic Cancer" in cancer_type_flags) :
                    demographics.append(f"{label}={value}")
                else:
                    continue
            elif label in attr_whitelist[src]:
                clinical_attrs.append(f"{label}={value}")

    blocks = []
    if demographics:
        blocks.append("Demographics: " + "; ".join(sorted(demographics)))
    if clinical_attrs:
        blocks.append("Clinical Attributes: " + "; ".join(sorted(clinical_attrs)))
    if biomarkers:
        blocks.append("Biomarkers: " + "; ".join(sorted(biomarkers)))
    if tumor_sites:
        blocks.append("Tumor Sites: " + ", ".join(sorted(tumor_sites)))

    treatments = record.get("Treatment", [])
    extra_therapy = record.get("TREATMENT", [])
    all_radiation = [t for t in (treatments + extra_therapy) if "Radiation" in t.get("SUBTYPE", "")]
    radiation = "; ".join(f"{t.get('SUBTYPE')} start={t.get('START_DATE')}" for t in all_radiation) or "None"

    treatment_lines = [
        extract_treatment_summary(treatments, "chemo", "Chemotherapy"),
        extract_treatment_summary(treatments, "immuno", "Immunotherapy"),
        extract_treatment_summary(treatments, "investigational", "Investigational Treatments"),
        f"- Radiation: {radiation}",
        extract_treatment_summary(treatments, "targeted", "Targeted Treatments")
    ]
    blocks.append("Treatments:\n" + "\n".join(treatment_lines))

    if (lab := summarize_lab_tests(record.get("LAB_TEST", []), cancer_type_flags)):
        blocks.append(lab)

    if len(cancer_type_flags) > 1:
        blocks.append("Diagnoses: " + ", ".join(sorted(cancer_type_flags)))

    if sample_data:
        cancer_type_to_samples = defaultdict(list)
        for sid, attrs in sample_data.items():
            ct = attrs.get("Cancer Type", "Unknown Cancer")
            cancer_type_to_samples[ct].append((sid, attrs))

        for ct, samples in cancer_type_to_samples.items():
            for sid, attrs in samples:
                blocks.append(f"Sample-Specific Information ({ct}):")
                selected = {k: v for k, v in attrs.items() if k in sample_attrs_by_cancer[ct]}
                for k, v in selected.items():
                    blocks.append(f"- {k}: {v}")

    patient_summary = "\n".join(blocks)

    if len(cancer_type_flags) == 1:
        (single_cancer,) = cancer_type_flags
        task_note = cancer_prompt_templates.get(single_cancer, f"The patient has been diagnosed with {single_cancer}. Predict survival using all relevant attributes.")
    else:
        task_note = (
            "If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer "
            "based on available stage, treatment history, or metastasis patterns."
        )

    prompt = (
        "You are an advanced cancer clinical outcome prediction model\n\n"
        "### TASK\n"
        "Based on the structured patient data below, assess the likelihood of survival and predict the cancer clinical treatment outcome.\n"
        f"{task_note}\n\n"
        "### PATIENT DATA SUMMARY\n"
        f"{patient_summary}\n\n"
        "### OUTPUT FORMAT\n"
        "Please respond in the following format:\n"
        "- Overall Survival Status ('0:LIVING' or '1:DECEASED')\n"
        "- Estimated Overall Survival in Months (float value)"
    )


    return prompt, survival_status, survival_months




with open("patient_data_final/P-0000205.json") as f:
    record = json.load(f)

prompt, status, months = generate_patient_prompt(record)
print(prompt)


You are an advanced cancer clinical outcome prediction model

### TASK
Based on the structured patient data below, assess the likelihood of survival and predict the cancer clinical treatment outcome.
The patient has Non-Small Cell Lung Cancer. Focus on relevant clinical factors such as Cancer Stage, Age, smoking history, and metastasis patterns.

### PATIENT DATA SUMMARY
Demographics: Current Age=56; Sex=Female
Clinical Attributes: Cancer Stage 1-3; Smoking=Unknown
Tumor Sites: Adrenal Glands, Bone, CNS/Brain, Liver, Lung, Lymph Node, Other, Pleura
Treatments:
- Chemotherapy: CISPLATIN, PEMETREXED, Days: Unknown

- Investigational Treatments: INVESTIGATIONAL, Days: 58–58
- Radiation: Radiation Therapy start=-334; Radiation Therapy start=-298; Radiation Therapy start=86; Radiation Therapy start=98
- Targeted Treatments: ERLOTINIB, Days: 110–111
Sample-Specific Information (Non-Small Cell Lung Cancer):
- Clinical Group: 3A
- Cancer Type Detailed: Lung Adenocarcinoma
- Sample Type: Metast

In [24]:
import json
from collections import defaultdict
from scipy.stats import linregress
import statistics

# === Load attribute metadata ===
with open("attributes_description.json") as f:
    attr_meta = json.load(f)

attr_priority = {}
attr_source = {}
for attr in attr_meta:
    label = attr["displayName"].replace("(NLP)", "").strip()
    priority = int(attr["priority"])
    attr_priority[label] = "HIGH" if priority >= 900 else "MEDIUM" if priority >= 500 else "LOW"
    attr_source[label] = "Patient" if attr["patientAttribute"] else "Sample"

# === Define cancer-specific keywords and attribute mappings ===
cancer_specific_keywords = {
    "Breast Cancer": {"HER2", "ER", "PR", "HR", "PD-L1"},
    "Colorectal Cancer": {"MSI", "TMB", "Mutation Count"},
    "Non-Small Cell Lung Cancer": {"PD-L1", "EGFR", "ALK", "Smoking History (NLP)", "Smoking History"},
    "Pancreatic Cancer": {"MSI", "TMB"},
    "Prostate Cancer": {"Gleason", "PSA"},
}

cancer_type_attr_map = defaultdict(lambda: {"Patient": set(), "Sample": set()})
for attr, prio in attr_priority.items():
    for cancer, keywords in cancer_specific_keywords.items():
        if any(k in attr for k in keywords) or attr in {
            "Cancer Type", "Cancer Type Detailed", "Stage (Highest Recorded)",
            "Sex", "Current Age", "Overall Survival Status", "Smoking History","Smoking History (NLP)"
        }:
            cancer_type_attr_map[cancer][attr_source[attr]].add(attr)

# === Sample attributes for each cancer ===
sample_attrs_by_cancer = defaultdict(set)
sample_attrs_by_cancer.update({
    "Breast Cancer": {"Cancer Type Detailed", "Sample Type", "Metastatic Site", "Clinical Summary"},
    "Non-Small Cell Lung Cancer": {"Cancer Type Detailed", "Sample Type", "Metastatic Site", "MSI Type", "MSI Score", "Clinical Group"},
    "Colorectal Cancer": {"Clinical Summary", "Diagnosis Description", "Cancer Type Detailed", "Sample Type", "Clinical Group", "Metastatic Site", "MSI Type","MSI Score", "Primary Tumor Site"},
    "Prostate Cancer": {"Clinical Summary", "Cancer Type Detailed", "Sample Type", "Clinical Group", "Metastatic Site","Gleason Score Reported on Sample"},
    "Pancreatic Cancer": {"Clinical Summary", "Cancer Type Detailed", "Sample Type", "Metastatic Site", "MSI Type", "MSI Score","Clinical Group"}
})

# === Summarize lab tests ===
from scipy.stats import linregress

def summarize_lab_tests(lab_tests, cancer_types):
    def compute_stats(values):
        if len(values) < 2:
            return {
                "avg": round(values[0][1], 1),
                "note": "single value"
            }

        values.sort()
        days, results = zip(*values)

        if len(set(days)) == 1:
            return {
                "avg": round(statistics.mean(results), 1),
                "note": "multiple results from same day"
            }

        slope, *_ = linregress(days, results)
        trend = "rising" if slope > 0 else "falling" if slope < 0 else "stable"

        return {
            "avg": round(statistics.mean(results), 1),
            "trend": trend,
            "slope": round(slope, 3),
            "start": results[0],
            "end": results[-1],
            "peak": max(results),
            "days": days[-1] - days[0]
        }

    def extract_values(aliases):
        return [(int(entry.get("DAYS_FROM_DIAGNOSIS", entry.get("START_DATE", 0))), float(entry["RESULT"]))
                for entry in lab_tests
                if entry.get("TEST", "").strip().upper() in aliases
                and entry.get("RESULT", "").replace('.', '', 1).isdigit()]

    markers = []
    tests = {
        "CEA": {"alias": {"CEA"}, "cancers": {"Colorectal Cancer", "Pancreatic Cancer", "Non-Small Cell Lung Cancer", "Breast Cancer"}},
        "CA15-3": {"alias": {"CA_15-3", "CA15-3", "CA 15-3"}, "cancers": {"Breast Cancer"}},
        "CA19-9": {"alias": {"CA_19-9", "CA19-9"}, "cancers": {"Colorectal Cancer", "Pancreatic Cancer"}},
        "PSA": {"alias": {"PSA"}, "cancers": {"Prostate Cancer"}}
    }

    for marker, config in tests.items():
        if not cancer_types.intersection(config["cancers"]):
            continue
        values = extract_values(config["alias"])
        if values:
            stats = compute_stats(values)
            if "start" in stats:
                markers.append(
                    f"- {marker}: {stats['start']} → {stats['end']} over {stats['days']} days ({stats['trend']}); "
                    f"avg={stats['avg']}; peak={stats['peak']}"
                )
            elif "note" in stats:
                markers.append(f"- {marker}: {stats['avg']} ({stats['note']})")
            else:
                markers.append(f"- {marker}: {stats['avg']}")

    return "Key Tumor Markers:\n" + "\n".join(markers) if markers else ""




# === Treatment summarization ===
def extract_treatment_summary(events, type_key, label):
    lines = []
    dates = []
    for t in events:
        if t.get("SUBTYPE", "").lower() == type_key:
            agent = t.get("AGENT", "").replace("(NLP)", "").strip()
            start, stop = t.get("START_DATE"), t.get("STOP_DATE")
            dates.append((start, stop))
            lines.append(agent.upper() if agent else "[Unknown Agent]")
    if not lines:
        return ""
    agents = ", ".join(sorted(set(lines)))
    all_dates = [int(d) for pair in dates for d in pair if d and str(d).isdigit()]
    date_range = f"Days: {min(all_dates)}–{max(all_dates)}" if all_dates else "Days: Unknown"
    return f"- {label}: {agents}, {date_range}"

# === Main patient summary generator ===
def extract_patient_info(record):
    clinical = record.get("CLINICAL_DATA", [])
    patient_id = record.get("patient_id", [])
    survival_status = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival Status"), "N/A")
    survival_months = next((i["Value"] for i in clinical if i["Attribute"] == "Overall Survival (Months)"), "N/A")

    cancer_types = set()
    sample_data = defaultdict(dict)
    for item in clinical:
        attr = item.get("Attribute", "").replace("(NLP)", "").strip()
        for k, v in item.items():
            if k.startswith("P-") and "-T" in k and v and v.lower() != "n/a":
                sample_data[k][attr] = v
                if attr == "Cancer Type":
                    cancer_types.add(v)

    clinical_attrs, tumor_sites, biomarkers =  [], set(), []
    for item in clinical:
        label = item.get("Attribute", "").replace("(NLP)", "").strip()
        value = item.get("Value", "").replace("(NLP)", "").strip()
        if not value or value.lower() == "n/a" or label in {
            "Overall Survival Status", "Overall Survival (Months)",
            "Number of Tumor Registry Entries", "Number of Samples Per Patient",
            "Race", "Ethnicity"
        }:
            continue
        if label in {"HER2", "PD-L1", "HR", "ER", "PR"}:
            if label == "PD-L1" or ("Breast Cancer" in cancer_types and label != "PD-L1"):
                biomarkers.append(f"{label}={value}")
        elif "Tumor Site:" in label and value.lower() in {"yes", "true"}:
            tumor_sites.add(label.split("Tumor Site:")[-1].strip())
        elif label == "Smoking History":
            if "Non-Small Cell Lung Cancer" in cancer_types:
                clinical_attrs.append(f"Smoking History={value}")
            continue
        elif label == "Stage (Highest Recorded)":
            clinical_attrs.append(f"Cancer {value}")
        elif label in {"Sex"}:
            if ("Non-Small Cell Lung Cancer" in cancer_types or "Colorectal Cancer" in cancer_types or "Pancreatic Cancer" in cancer_types) :
                clinical_attrs.append(f"{label}={value}")
            else:
                continue
        else:
            src = attr_source.get(label, "Patient")
            if label in cancer_type_attr_map[next(iter(cancer_types), "")][src]:
                clinical_attrs.append(f"{label}={value}")
                
    blocks = []
    if clinical_attrs:
        blocks.append("Clinical Attributes: " + "; ".join(sorted(clinical_attrs)))
    if biomarkers:
        blocks.append("Biomarkers: " + "; ".join(sorted(biomarkers)))
    if tumor_sites:
        blocks.append("Tumor Sites: " + ", ".join(sorted(tumor_sites)))


    treatments = record.get("Treatment", [])
    extra_therapy = record.get("TREATMENT", [])
    all_radiation = [t for t in (treatments + extra_therapy) if "Radiation" in t.get("SUBTYPE", "")]

    radiation_dates = [
        int(t.get("START_DATE"))
        for t in all_radiation
        if t.get("START_DATE") not in [None, "", "N/A"] and str(t.get("START_DATE")).lstrip('-').isdigit()
    ]

    radiation_summary = ""
    if radiation_dates:
        sorted_dates = sorted(radiation_dates)
        radiation_summary = f"- Radiation Therapy: Days {', '.join(str(d) for d in sorted_dates)}"

    treatment_lines = [
        extract_treatment_summary(treatments, "chemo", "Chemotherapy"),
        extract_treatment_summary(treatments, "immuno", "Immunotherapy"),
        extract_treatment_summary(treatments, "investigational", "Investigational Treatments"),
        extract_treatment_summary(treatments, "targeted", "Targeted Treatments")
    ]

    # Add radiation line only if there are valid dates
    if radiation_summary:
        treatment_lines.append(radiation_summary)


    # Append treatments block
    blocks.append("Treatments:\n" + "\n".join([line for line in treatment_lines if line]))


    if (lab := summarize_lab_tests(record.get("LAB_TEST", []), cancer_types)):
        blocks.append(lab)

    if len(cancer_types) > 1:
        blocks.append("Diagnoses: " + ", ".join(sorted(cancer_types)))

    # if sample_data:
    #     cancer_type_to_samples = defaultdict(list)
    #     for sid, attrs in sample_data.items():
    #         ct = attrs.get("Cancer Type", "Unknown Cancer")
    #         cancer_type_to_samples[ct].append((sid, attrs))

    #     for ct, samples in cancer_type_to_samples.items():
    #         for sid, attrs in samples:
    #             blocks.append(f"Sample-Specific Information ({ct}):")
    #             selected = {k: v for k, v in attrs.items() if k in sample_attrs_by_cancer[ct]}
    #             for k, v in selected.items():
    #                 blocks.append(f"- {k}: {v}")
    if sample_data:
        cancer_type_to_samples = defaultdict(list)
        for sid, attrs in sample_data.items():
            ct = attrs.get("Cancer Type", "Unknown Cancer")
            cancer_type_to_samples[ct].append((sid, attrs))

        for ct, samples in cancer_type_to_samples.items():
            for sid, attrs in samples:
                blocks.append(f"Sample-Specific Information ({ct}):")
                selected = {}

                for k, v in attrs.items():
                # Special filtering for ICD-O Histology Description
                    if k == "ICD-O Histology Description":
                        if ct in {"Colorectal Cancer", "Pancreatic Cancer", "Prostate Cancer"}:
                            if v not in {"Adenocarcinoma, Nos", "Carcinoma, Nos"}:
                                selected[k] = v
                    elif k in sample_attrs_by_cancer[ct]:
                        selected[k] = v

                for k, v in selected.items():
                    blocks.append(f"- {k}: {v}")


    patient_summary = "\n".join(blocks)

    # return patient_id, sorted(cancer_types), survival_status, survival_months, patient_summary
    return {
            "patient_data": patient_summary,
            "survival_status": survival_status,
            "survival_months": survival_months,
            "cancer_type": sorted(cancer_types),
            "patient_id": patient_id
        }

def generate_prompt(patient_summary: str, cancer_types: list) -> str:
    # Task note template based on cancer type
    task_notes = {
        "Breast Cancer": (
            "The patient has been diagnosed with Breast Cancer. "
            "Use your clinical reasoning to interpret the data attributes related to Cancer Stage, Age, tumor biology, treatments, biomarkers, and metastasis patterns."
        ),
        "Non-Small Cell Lung Cancer": (
            "The patient has Non-Small Cell Lung Cancer. "
            "Focus on relevant clinical factors such as Cancer Stage, Age, smoking history, and metastasis patterns."
        ),
        "Colorectal Cancer": (
            "The patient has Colorectal Cancer. "
            "Focus on relevant clinical factors such as Cancer Stage, Age, tumor markers, treatment data, and metastasis patterns."
        ),
        "Pancreatic Cancer": (
            "The patient has Pancreatic Cancer. "
            "Consider age, stage and relevant key factors for prognosis."
        ),
        "Prostate Cancer": (
            "The patient has Prostate Cancer. "
            "Focus on relevant key factors such as Cancer Stage, Age, Gleason score, PSA levels, and treatment data."
        ),
    }

    # Select appropriate task note
    if len(cancer_types) == 1:
        task_note = task_notes.get(
            cancer_types[0],
            f"The patient has been diagnosed with {cancer_types[0]}. Predict survival using all relevant attributes.",
        )
    else:
        task_note = (
            "If the patient has multiple cancer types, focus on the most clinically aggressive or recently treated cancer "
            "based on available stage, age, treatment history, or metastasis patterns."
        )

    # Generate prompt
    prompt = (
        "You are an advanced clinical reasoning model specialized in predicting cancer treatment outcomes.\n\n"
        "### TASK\n"
        "Based on the structured patient data below, assess the likelihood of survival and predict the cancer clinical treatment outcome.\n"
        f"{task_note}\n\n"
        "### PATIENT DATA SUMMARY\n"
        f"{patient_summary}\n\n"
        "### OUTPUT FORMAT\n"
        "Please respond in the following format:\n"
        "- Overall Survival Status: '0:LIVING' or '1:DECEASED'\n"
        "- Estimated Overall Survival (in months): float value"
    )

    return prompt


with open("patient_data_final/P-0000015.json") as f:
    record = json.load(f)

patient, cancer, status, months, patient_data = extract_patient_info(record)
print(patient_data)


patient_id


In [25]:
import os
import json

# Import your generate_patient_summary function here
# from your_module import generate_patient_summary

input_folder = "patient_data_final"
output_file = "patient_summary_prompt_data.json"

all_results = []

for filename in os.listdir(input_folder):
    if filename.endswith(".json"):
        file_path = os.path.join(input_folder, filename)
        with open(file_path) as f:
            record = json.load(f)
        try:
            result = extract_patient_info(record)
            all_results.append(result)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            
all_results.sort(key=lambda x: x.get("patient_id", ""))
# Save to one JSON file
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)


In [50]:
print("Clinical Attributes: Cancer Stage 1-3; Current Age=68; History for Positive PD-L1=No; Sex=Female; Smoking History=Former/Current Smoker\nBiomarkers: HER2=No; HR=No\nTumor Sites: Intra Abdominal, Lung, Lymph Node, Other\nTreatments:\n- Chemotherapy: CARBOPLATIN, CISPLATIN, CYCLOPHOSPHAMIDE, ETOPOSIDE, FLUOROURACIL, METHOTREXATE, Days: 33\u201368\n- Immunotherapy: NIVOLUMAB, Days: 1734\u20131814\n- Investigational Treatments: INVESTIGATIONAL, Days: 320\u2013320\n- Radiation Therapy: Days 33, 1721\nKey Tumor Markers:\n- CEA: 1.8 \u2192 4.6 over 5354 days (rising); avg=2.1; peak=4.6\n- CA15-3: 21.0 \u2192 13.0 over 5224 days (falling); avg=13.3; peak=21.0\nDiagnoses: Breast Cancer, Non-Small Cell Lung Cancer\nSample-Specific Information (Non-Small Cell Lung Cancer):\n- Clinical Group: 3B\n- Cancer Type Detailed: Lung Adenocarcinoma\n- Sample Type: Metastasis\n- MSI Type: Stable\n- Metastatic Site: Neck\n- MSI Score: 0.47\nSample-Specific Information (Breast Cancer):\n- Cancer Type Detailed: Breast Invasive Ductal Carcinoma\n- Sample Type: Primary")

Clinical Attributes: Cancer Stage 1-3; Current Age=68; History for Positive PD-L1=No; Sex=Female; Smoking History=Former/Current Smoker
Biomarkers: HER2=No; HR=No
Tumor Sites: Intra Abdominal, Lung, Lymph Node, Other
Treatments:
- Chemotherapy: CARBOPLATIN, CISPLATIN, CYCLOPHOSPHAMIDE, ETOPOSIDE, FLUOROURACIL, METHOTREXATE, Days: 33–68
- Immunotherapy: NIVOLUMAB, Days: 1734–1814
- Investigational Treatments: INVESTIGATIONAL, Days: 320–320
- Radiation Therapy: Days 33, 1721
Key Tumor Markers:
- CEA: 1.8 → 4.6 over 5354 days (rising); avg=2.1; peak=4.6
- CA15-3: 21.0 → 13.0 over 5224 days (falling); avg=13.3; peak=21.0
Diagnoses: Breast Cancer, Non-Small Cell Lung Cancer
Sample-Specific Information (Non-Small Cell Lung Cancer):
- Clinical Group: 3B
- Cancer Type Detailed: Lung Adenocarcinoma
- Sample Type: Metastasis
- MSI Type: Stable
- Metastatic Site: Neck
- MSI Score: 0.47
Sample-Specific Information (Breast Cancer):
- Cancer Type Detailed: Breast Invasive Ductal Carcinoma
- Sample T

In [ ]:
import json

# Load from file
with open("cot_final.json", "r") as f:
    patient_data = json.load(f)

# Sort by numeric part of patient ID
sorted_data = dict(sorted(patient_data.items(), key=lambda x: int(x[0].split("-")[1])))

# Save back to file (optional)
with open("cot_sorted.json", "w") as f:
    json.dump(sorted_data, f, indent=4)

# Optionally print it
print(json.dumps(sorted_data, indent=4))


In [51]:
print("{\n    \"chain_of_thought\": [\n        \"Step 1: Assess cancer stage and type. Stage 3B NSCLC is locally advanced with poor prognosis, but multimodal therapy can improve outcomes. Tumor spread to intra-abdominal, lymph node, and other sites suggests aggressive disease but may still be regional in stage 3B.\",\n        \"Step 2: Evaluate patient-specific factors. Age 68 and smoking history may reduce treatment tolerance and increase comorbidity risks, but the patient received multiple therapies, indicating adequate functional status.\",\n        \"Step 3: Analyze chemotherapy regimens. Early non-standard agents (e.g., cyclophosphamide, fluorouracil) may reflect clinical trial participation or prior treatment for another condition. Later platinum-based combinations (cisplatin/carboplatin + etoposide) align with stage 3B NSCLC protocols, suggesting definitive chemoradiation intent.\",\n        \"Step 4: Consider immunotherapy timing. Nivolumab administered ~5 years after initial therapy likely represents second-line or maintenance treatment, potentially prolonging control in PD-L1-positive cases, though biomarker status is unknown.\",\n        \"Step 5: Investigational treatments at day 320 introduce uncertainty but may indicate early trial participation, possibly targeting residual disease post-chemoradiation.\",\n        \"Step 6: Radiation at days 33 and 1721 suggests initial definitive chemoradiation followed by salvage or consolidative therapy, consistent with stage 3B management. Repeat radiation may target oligoprogression.\",\n        \"Step 7: Synthesize factors. Aggressive multimodal therapy (chemoradiation, immunotherapy, trials), disease confined to regional sites (stage 3B), and tolerance to prolonged treatment suggest potential for sustained disease control despite high-risk features.\"\n    ],\n    \"comments\": \"Uncertainties include PD-L1 status, investigational treatment details, genetic alterations (e.g., EGFR/ALK), and toxicity management. Smoking history\u2019s impact on immunotherapy efficacy is unclear. Repeat radiation timing (day 1721) could indicate recurrence or consolidation, altering prognosis.\"\n}")

{
    "chain_of_thought": [
        "Step 1: Assess cancer stage and type. Stage 3B NSCLC is locally advanced with poor prognosis, but multimodal therapy can improve outcomes. Tumor spread to intra-abdominal, lymph node, and other sites suggests aggressive disease but may still be regional in stage 3B.",
        "Step 2: Evaluate patient-specific factors. Age 68 and smoking history may reduce treatment tolerance and increase comorbidity risks, but the patient received multiple therapies, indicating adequate functional status.",
        "Step 3: Analyze chemotherapy regimens. Early non-standard agents (e.g., cyclophosphamide, fluorouracil) may reflect clinical trial participation or prior treatment for another condition. Later platinum-based combinations (cisplatin/carboplatin + etoposide) align with stage 3B NSCLC protocols, suggesting definitive chemoradiation intent.",
        "Step 4: Consider immunotherapy timing. Nivolumab administered ~5 years after initial therapy likely represe

In [47]:
print("Clinical Attributes: Cancer Stage 1-3; Current Age=45\nBiomarkers: HER2=No; HR=Yes\nTumor Sites: Bone, CNS/Brain, Liver, Lung, Lymph Node, Other, Pleura\nTreatments:\n- Chemotherapy: CAPECITABINE, CARBOPLATIN, CISPLATIN, EPIRUBICIN, GEMCITABINE, PACLITAXEL, VINORELBINE, Days: 15\u2013415\n- Investigational Treatments: INVESTIGATIONAL, Days: Unknown\n- Radiation Therapy: Days -1, 0, 328, 335\nKey Tumor Markers:\n- CEA: 10.6 \u2192 26.8 over 722 days (falling); avg=16.9; peak=40.6\n- CA15-3: 49.0 \u2192 165.0 over 712 days (rising); avg=146.8; peak=400.0\nSample-Specific Information (Breast Cancer):\n- Cancer Type Detailed: Breast Invasive Ductal Carcinoma\n- Sample Type: Metastasis\n- Metastatic Site: Liver")

Clinical Attributes: Cancer Stage 1-3; Current Age=45
Biomarkers: HER2=No; HR=Yes
Tumor Sites: Bone, CNS/Brain, Liver, Lung, Lymph Node, Other, Pleura
Treatments:
- Chemotherapy: CAPECITABINE, CARBOPLATIN, CISPLATIN, EPIRUBICIN, GEMCITABINE, PACLITAXEL, VINORELBINE, Days: 15–415
- Investigational Treatments: INVESTIGATIONAL, Days: Unknown
- Radiation Therapy: Days -1, 0, 328, 335
Key Tumor Markers:
- CEA: 10.6 → 26.8 over 722 days (falling); avg=16.9; peak=40.6
- CA15-3: 49.0 → 165.0 over 712 days (rising); avg=146.8; peak=400.0
Sample-Specific Information (Breast Cancer):
- Cancer Type Detailed: Breast Invasive Ductal Carcinoma
- Sample Type: Metastasis
- Metastatic Site: Liver


In [52]:
print("Stage: Stage 1-3.\nAge: 68. Smoking History: Former/Current Smoker.\nTumor Sites: Intra Abdominal, Lung, Lymph Node, Other.\nChemotherapy: CYCLOPHOSPHAMIDE (-5437 to -5369); FLUOROURACIL (-5437 to -5326); METHOTREXATE (-5437 to -5327); CISPLATIN (33 to 40); ETOPOSIDE (33 to 65); CARBOPLATIN (61 to 68).\nImmunotherapy: NIVOLUMAB (1734 to 1814).\nInvestigational Treatments: INVESTIGATIONAL (320 to 320); INVESTIGATIONAL (320 to 320).\nRadiation Therapy: Radiation Therapy starting 33; Radiation Therapy starting 1721.\nCancer Type: Non-Small Cell Lung Cancer. Clinical Group: 3B.\nPrimary Tumor Site: Lung.\n")

Stage: Stage 1-3.
Age: 68. Smoking History: Former/Current Smoker.
Tumor Sites: Intra Abdominal, Lung, Lymph Node, Other.
Chemotherapy: CYCLOPHOSPHAMIDE (-5437 to -5369); FLUOROURACIL (-5437 to -5326); METHOTREXATE (-5437 to -5327); CISPLATIN (33 to 40); ETOPOSIDE (33 to 65); CARBOPLATIN (61 to 68).
Immunotherapy: NIVOLUMAB (1734 to 1814).
Investigational Treatments: INVESTIGATIONAL (320 to 320); INVESTIGATIONAL (320 to 320).
Radiation Therapy: Radiation Therapy starting 33; Radiation Therapy starting 1721.
Cancer Type: Non-Small Cell Lung Cancer. Clinical Group: 3B.
Primary Tumor Site: Lung.



In [48]:
print("Stage: Stage 1-3.\nAge: 45. Smoking History: Unknown.\nTumor Sites: Bone, CNS/Brain, Liver, Lung, Lymph Node, Other, Pleura.\nChemotherapy: None.\nImmunotherapy: None.\nInvestigational Treatments: None.\nRadiation Therapy: None.\nCancer Type: Breast Cancer. Clinical Group: 1.\nPrimary Tumor Site: Breast.\n")

Stage: Stage 1-3.
Age: 45. Smoking History: Unknown.
Tumor Sites: Bone, CNS/Brain, Liver, Lung, Lymph Node, Other, Pleura.
Chemotherapy: None.
Immunotherapy: None.
Investigational Treatments: None.
Radiation Therapy: None.
Cancer Type: Breast Cancer. Clinical Group: 1.
Primary Tumor Site: Breast.



In [180]:
print("Stage: Stage 4.\nAge: 45. Smoking History: Unknown.\nTumor Sites: Bone, Intra Abdominal, Liver, Lung, Lymph Node, Other, Reproductive Organs.\nChemotherapy: None.\nImmunotherapy: None.\nInvestigational Treatments: None.\nRadiation Therapy: Radiation Therapy starting -122.\nCancer Type: Prostate Cancer. Clinical Group: 4.\nPrimary Tumor Site: Prostate.\n")

Stage: Stage 4.
Age: 45. Smoking History: Unknown.
Tumor Sites: Bone, Intra Abdominal, Liver, Lung, Lymph Node, Other, Reproductive Organs.
Chemotherapy: None.
Immunotherapy: None.
Investigational Treatments: None.
Radiation Therapy: Radiation Therapy starting -122.
Cancer Type: Prostate Cancer. Clinical Group: 4.
Primary Tumor Site: Prostate.

